# EDA

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
_ROOT_DIR = "./../"

In [ ]:
# main data 불러오기
def get_main_df(): 
    df_main = pd.read_csv(_ROOT_DIR+"./data/raw/main.csv")
    return df_main

In [ ]:
# target data 불러오기
def get_target_df():
    df_test = pd.read_csv(_ROOT_DIR+"./data/raw/test_cxid.csv")
    df_train = pd.read_csv(_ROOT_DIR+"./data/raw/train_cxid.csv")
    df_target = pd.concat([df_test, df_train], axis=0)
    return df_target

In [ ]:
def merge_main_target(df_main):
    df_target = get_target_df()
    df_merged = pd.merge(left=df_main, right=df_target, how="inner", on="customer_id")
    return df_merged

In [ ]:
# 사용량이 없는 경우 행 존재 여부 확인
def check_na_count():
    df = get_main_df()
    print(df['usage_type'].value_counts())
    return df

In [ ]:
# 행/열 pivot
def pivot_df(df):
    # days 평균 계산 (행 방향 평균)
    day_cols = [col for col in df.columns if col.startswith('Day_')]
    df['avg_usage'] = df[day_cols].mean(axis=1)

    # 행 열 변경
    df_pivot = df.pivot_table(
        index='customer_id', 
        columns='usage_type', 
        values='avg_usage', 
        aggfunc='mean' 
    )

    # columns.name, index 제거 
    df_pivot.columns.name = None
    df_pivot = df_pivot.reset_index()

    return df_pivot

In [ ]:
# 통화량 feature 값 수신량/발신량으로 통합하기
def integrate_feat_voice(df):
    # 수신 칼럼 합치기
    df['usage_voice_incoming'] = df['usage_voice_d2d_incoming']+df['usage_voice_nd2d_incoming']
    df.drop(columns=['usage_voice_d2d_incoming', 'usage_voice_nd2d_incoming'], inplace=True)
    # 발신 칼럼 합치기
    df['usage_voice_outgoing'] = df['usage_voice_d2d_outgoing']+df['usage_voice_d2nd_outgoing']
    df.drop(columns=['usage_voice_d2d_outgoing', 'usage_voice_d2nd_outgoing'], inplace=True)
    return df

In [ ]:
# app 관련 feature 파생변수 생성
def derive_feat_app(df):
    # 'usage_app'으로 시작하는 모든 칼럼들 평균
    df['avg_usage_app'] = df.filter(like='usage_app').mean(axis=1)
    return df

In [ ]:
# voice 관련 feature 파생변수 생성
def derive_feat_voice(df):
    df['avg_usage_voice'] = df.filter(like='usage_voice').mean(axis=1)
    return df

In [ ]:
def get_df():
    df = get_main_df()
    df = pivot_df(df)
    df = integrate_feat_voice(df)
    df = derive_feat_voice(df)
    df = derive_feat_app(df)
    df = merge_main_target(df)
    return df

In [ ]:
df = get_df()
df.info()